In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence

from datasets import load_dataset
from collections import Counter
import json

from sklearn.metrics import f1_score, precision_score, recall_score

import numpy as np
import pandas as pd
from tqdm import tqdm
import random

In [2]:
dataset = load_dataset("lhoestq/conll2003")

label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

print(f"Labels: {label_list}")
print(f"Train sentences: {len(dataset['train'])}")
print(f"Val sentences:   {len(dataset['validation'])}")
print(f"Test sentences:  {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Train sentences: 14041
Val sentences:   3250
Test sentences:  3453


In [3]:
# — Vocabulary Building
def build_vocab(dataset, min_freq=1, lowercase=True):

    counter = Counter()

    for example in dataset['train']:
        if lowercase:
            tokens = [word.lower() for word in example['tokens']]
        else:
            tokens = example['tokens']
        counter.update(tokens)

    vocab = {'<PAD>': 0, '<UNK>': 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab = build_vocab(dataset, min_freq=2, lowercase=True)

print("Building vocabulary...")
print(f"Vocabulary size: {len(vocab):,}")
print(f"  '<PAD>' -> {vocab['<PAD>']}")
print(f"  '<UNK>' -> {vocab['<UNK>']}")
print(f"  'london' -> {vocab.get('london', 'Not found')}")

Building vocabulary...
Vocabulary size: 10,951
  '<PAD>' -> 0
  '<UNK>' -> 1
  'london' -> 247


In [4]:
class NERDataset(Dataset):

  def __init__(self, data_split, vocab):
    self.data_split = data_split
    self.vocab = vocab

  def __len__(self):
    return len(self.data_split)

  def __getitem__(self, idx):
    tokens = self.data_split[idx]['tokens']
    labels = self.data_split[idx]['ner_tags']

    input_ids = [self.vocab.get(token.lower(), self.vocab['<UNK>']) for token in tokens]

    return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)


def collate_fn(batch):
    inputs, labels = zip(*batch)
    padded_inputs = pad_sequence(inputs, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return padded_inputs, padded_labels


# ── Hyperparameters ─────────────────────────────────────────
BATCH_SIZE = 32

# ── Datasets Instantiation ──────────────────────────────────
train_dataset = NERDataset(dataset['train'],      vocab)
val_dataset   = NERDataset(dataset['validation'], vocab)
test_dataset  = NERDataset(dataset['test'],       vocab)

# ── DataLoaders Instantiation ───────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# ── Verification ──────────────────────────────────────
inputs, labels = next(iter(train_loader))
print(f"Input shape : {inputs.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Labels sample: {labels[0].tolist()}")

Input shape : torch.Size([32, 38])
Labels shape: torch.Size([32, 38])
Labels sample: [0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [5]:
# — Hyperparameters
VOCAB_SIZE    = len(vocab)
EMBEDDING_DIM = 100
HIDDEN_DIM    = 256
NUM_LAYERS    = 2
NUM_CLASSES   = len(label_list)
DROPOUT       = 0.3
PAD_IDX       = 0

print(f"Vocab size   : {VOCAB_SIZE:,}")
print(f"Num classes  : {NUM_CLASSES}")
print(f"Embedding dim: {EMBEDDING_DIM}")
print(f"Hidden dim   : {HIDDEN_DIM}")

Vocab size   : 10,951
Num classes  : 9
Embedding dim: 100
Hidden dim   : 256


In [6]:
class BaseNERModel(nn.Module):

  def __init__(self, vocab_size, embedding_dim, hidden_dim,
              num_classes, num_layers, dropout, pad_idx, rnn_type="LSTM"):
      super().__init__()

      self.rnn_type = rnn_type

      self.Embedding = nn.Embedding(
          num_embeddings=vocab_size,
          embedding_dim=embedding_dim,
          padding_idx=pad_idx
      )

      rnn_dropout = dropout if num_layers > 1 else 0
      rnn_args = dict(
          input_size=embedding_dim,
          hidden_size=hidden_dim,
          num_layers=num_layers,
          dropout=rnn_dropout,
          batch_first=True,
          bidirectional=True
      )

      if rnn_type == "RNN":
          self.rnn = nn.RNN(**rnn_args)
      elif rnn_type == "GRU":
          self.rnn = nn.GRU(**rnn_args)
      else:
          self.rnn = nn.LSTM(**rnn_args)

      self.dropout = nn.Dropout(dropout)
      self.fc = nn.Linear(hidden_dim * 2, num_classes)

  def forward(self, x):

      embedded = self.dropout(self.Embedding(x))
      output, _ = self.rnn(embedded)
      logits = self.fc(self.dropout(output))

      return logits

print("BaseNERModel class defined successfully!")

for rnn_type in ["RNN", "LSTM", "GRU"]:
    model = BaseNERModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM,
                     NUM_CLASSES, NUM_LAYERS, DROPOUT,
                     PAD_IDX, rnn_type=rnn_type)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"{rnn_type:4s} — Parameters: {total_params:,}")

BaseNERModel class defined successfully!
RNN  — Parameters: 1,677,253
LSTM — Parameters: 3,409,861
GRU  — Parameters: 2,832,325


In [7]:
def compute_metrics(preds, labels):

    preds  = preds.cpu().numpy()  if torch.is_tensor(preds)  else preds
    labels = labels.cpu().numpy() if torch.is_tensor(labels) else labels

    f1        = f1_score(labels, preds, average='macro', zero_division=0)
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall    = recall_score(labels, preds, average='macro', zero_division=0)

    return {'f1': round(f1, 4), 'precision': round(precision, 4), 'recall': round(recall, 4)}

print("✅ compute_metrics defined successfully!")

✅ compute_metrics defined successfully!


In [9]:
def train_one_epoch(model, loader, optimizer, criterion, device):

    model.train()
    total_loss = 0
    all_preds  = []
    all_labels = []

    for batch in tqdm(loader, desc="Training", leave=False):

      inputs, labels = batch[0].to(device), batch[1].to(device)

      optimizer.zero_grad()
      logits = model(inputs)

      loss = criterion(logits.view(-1, logits.shape[-1]), labels.view(-1))
      loss.backward()

      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      optimizer.step()

      total_loss += loss.item()

      preds = torch.argmax(logits, dim=-1)
      mask  = labels != -100
      all_preds.append(preds[mask].detach().cpu())
      all_labels.append(labels[mask].detach().cpu())

    metrics = compute_metrics(torch.cat(all_preds, dim=0), torch.cat(all_labels, dim=0))

    return total_loss / len(loader), metrics

def evaluate(model, loader, criterion, device):

    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:

          inputs, labels = batch[0].to(device), batch[1].to(device)

          logits = model(inputs)
          loss = criterion(logits.view(-1, logits.shape[-1]), labels.view(-1))
          total_loss += loss.item()

          preds = torch.argmax(logits, dim=-1)
          mask  = labels != -100
          all_preds.append(preds[mask].cpu())
          all_labels.append(labels[mask].cpu())

    metrics = compute_metrics(torch.cat(all_preds, dim=0), torch.cat(all_labels, dim=0))

    return total_loss / len(loader), metrics

def run_training(model, model_name, train_loader, val_loader, optimizer, criterion, device, epochs=5):

    history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [],   'val_f1': [],
        'val_precision': [], 'val_recall': []
    }

    best_val_f1 = 0.0

    print(f"Starting training for {model_name} on {device}...")
    print("-" * 60)

    for epoch in range(epochs):
      train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
      val_loss, val_metrics = evaluate(model, val_loader, criterion, device)

      history['train_loss'].append(train_loss)
      history['val_loss'].append(val_loss)
      history['train_f1'].append(train_metrics['f1'])
      history['val_f1'].append(val_metrics['f1'])
      history['val_precision'].append(val_metrics['precision'])
      history['val_recall'].append(val_metrics['recall'])

      status = ""
      if val_metrics['f1'] > best_val_f1:
          best_val_f1 = val_metrics['f1']
          torch.save(model.state_dict(), f'best_{model_name}.pt')
          status = "Best Model Saved!"

      print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"Loss: {train_loss:.3f}/{val_loss:.3f} | "
              f"F1 (T/V): {train_metrics['f1']:.3f}/{val_metrics['f1']:.3f} | "
              f"Prec/Rec (V): {val_metrics['precision']:.3f}/{val_metrics['recall']:.3f} {status}")

    print("-" * 60)
    print(f"✅ Training Finished! Best Validation F1: {best_val_f1:.4f}")

    return history

print("✅ Professional run_training function defined successfully!")

✅ Professional run_training function defined successfully!


In [10]:
# =====================================================================
#  EXPERIMENT 2: Bi-RNN
# =====================================================================

config = {
    "rnn_type": "RNN",
    "embedding_dim": 100,
    "hidden_dim": 256,
    "num_layers": 2,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "epochs": 5,
    "batch_size": 32
}

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn)

print(f"--- Running Experiment: {config['rnn_type']} ---")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BaseNERModel(
    vocab_size=VOCAB_SIZE,        # This is constant (does not change)
    embedding_dim=config["embedding_dim"],
    hidden_dim=config["hidden_dim"],
    num_classes=NUM_CLASSES,      # This is constant (does not change)
    num_layers=config["num_layers"],
    dropout=config["dropout"],
    pad_idx=PAD_IDX,              # This is constant (does not change)
    rnn_type=config["rnn_type"]
).to(device)


criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])

model_name = f"{config['rnn_type']}_H{config['hidden_dim']}_L{config['num_layers']}"
history = run_training(
    model=model,
    model_name=config["rnn_type"],
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=config["epochs"]
)

--- Running Experiment: RNN ---
Starting training for RNN on cuda...
------------------------------------------------------------


Epoch 01/5 | Loss: 0.540/0.402 | F1 (T/V): 0.365/0.469 | Prec/Rec (V): 0.811/0.371 Best Model Saved!


Epoch 02/5 | Loss: 0.388/0.307 | F1 (T/V): 0.548/0.611 | Prec/Rec (V): 0.758/0.538 Best Model Saved!


Epoch 03/5 | Loss: 0.309/0.260 | F1 (T/V): 0.638/0.666 | Prec/Rec (V): 0.854/0.567 Best Model Saved!


Epoch 04/5 | Loss: 0.257/0.222 | F1 (T/V): 0.692/0.717 | Prec/Rec (V): 0.867/0.623 Best Model Saved!


Epoch 05/5 | Loss: 0.217/0.197 | F1 (T/V): 0.737/0.748 | Prec/Rec (V): 0.848/0.679 Best Model Saved!
------------------------------------------------------------
✅ Training Finished! Best Validation F1: 0.7479
